## **STATUS AS OF 11/11 10AM**

**What we have now**
* `music.db`: raw Gametime data, 1.9M daily rows
* `shows.csv`: 29,613 shows, each with a price about 14 days out
* `artists.csv`: 5,139 artists; 94% of shows have Last.fm popularity
* `venues.csv`:
  * 78% of venues matched to SeatGeek, covering 75% of shows
  * 1,943 venues in the US, 188 outside it, 591 unknown (unknown venues hold **25% of shows**)
  * **Capacity known for only 19% of shows**, and skewed toward big venues (345 large, 7 clubs)
  * County income known for 71% of venues
* `06_cleaning_procedure.ipynb`: has agenda and cleaning code (tasks 2–5)
* `venue_capacity_manual.csv`: 150 venues listed, **0 filled in**

**Next steps**
1. **Run 06** and read what each section prints.
2. **Decide on capacity** (biggest gap):
   * Hand-fill the 150 venues (about 40 each, then rerun 05), **or**
   * Drop capacity and lean on city, state and the SeatGeek venue score instead. With only 19% filled, capacity can't be a feature as it is now.
3. **Decide on unknown-country venues:** 25% of shows is too many to keep blindly. Spot-check the busiest unknown venues; most are probably US venues SeatGeek just missed.
4. **Lock the group decisions in 06:** target (`price_median` or `price_min`), price cap, and minimum listings (40% of shows have fewer than 3).
5. **Join everything into one table** at the end of 06, then **save `data/model_table.csv`**. 
6. 

| Person | Work | Why paired |
|---|---|---|
| **Model 1** | **OLS + Ridge/Lasso** | Same linear model with a penalty added; same feature prep, so one notebook |
| **Model 2** | **KNN + tree / random forest** | The two non-linear approaches; both get tuned the same way (k, depth) |
| **Model 3** | **PCA + K-means**, then test them inside Model 1's or Model 2's models | The required unsupervised part |
| **Dashboard 1** | **App build:** Dash layout and tabs, using dummy data, deployed to Render early | Deploying is the unfamiliar part; catch problems now, not Dec 5 |
| **Dashboard 2** | **Content:** the 3–4 EDA charts (used in both the app and slides), the app's text, and the slide outline | The charts are needed anyway, and they keep the app and deck consistent |

***

**basic cleaning was done when data was pulled and tabled.** this icnludes things like festival passes and theme nights dropped, artist names pulled from show names, one price per show (about 14 days out), and duplicates removed. 

THE FOUR TABLES (TO BE JOINED)

1. data/music.db      ← raw Gametime (1.9M daily rows), only used to build shows
2. data/shows.csv     ← one row per show (29,613)
3. data/artists.csv   ← one row per artist (5,139)
4. data/venues.csv    ← one row per venue (after 05 finishes)

**cleaning agenda**

- Hand-fill venue capacity (split the work): data/manual/venue_capacity_manual.csv, about 150 venues. Each person takes about 40, looking up capacity on the venue’s website or Wikipedia, then rerun 05.
- Check bad artist matches: 31 low-confidence Last.fm matches listed at the end of 04 (e.g., “World of Warcraft”, “Lord of the Rings”). Fix or drop them.
- Check unmatched venues: the busiest venues SeatGeek couldn’t find, listed at the end of 05. Look for typos or wrong cities.
- Remove non-US shows: about 1,000 shows in London, Montreal and similar; filter using SeatGeek’s country column.
- Extreme prices: shows priced above about $2,000 (the top is $49,907). Look at them and decide whether they’re real or junk.
- Group decisions:
    - target: lowest price or median price
    - minimum listings: 40% of shows have fewer than 3
    - any other show types to drop (e.g., Las Vegas magic and comedy shows like Criss Angel and Tape Face that aren’t concerts)

In [ ]:
# Setup: load the three tables
import numpy as np
import pandas as pd

shows = pd.read_csv("data/shows.csv", low_memory=False)
artists = pd.read_csv("data/artists.csv", low_memory=False)
venues = pd.read_csv("data/venues.csv", low_memory=False)
print(f"shows {len(shows):,} | artists {len(artists):,} | venues {len(venues):,}")

In [ ]:
# 2. Bad artist matches (Last.fm matched the wrong artist)
FIX_BAD_MATCHES = True   # True = blank out Last.fm data for low-confidence matches

bad = artists["low_confidence_match"].fillna(False).astype(bool)
print(f"{bad.sum()} low-confidence Last.fm matches:")
display(artists.loc[bad, ["artist_name", "lastfm_name", "name_similarity", "listeners", "n_shows"]]
               .sort_values("n_shows", ascending=False))

if FIX_BAD_MATCHES:
    lastfm_cols = ["lastfm_name", "listeners", "log_listeners", "playcount",
                   "plays_per_listener", "on_tour", "tags", "top_tag"]
    artists.loc[bad, lastfm_cols] = np.nan       # keep the artist, drop the wrong popularity data
    artists.loc[bad, "lastfm_status"] = "bad_match"
    print(f"Blanked Last.fm data for {bad.sum()} artists")

In [ ]:
# 3. Unmatched venues (SeatGeek couldn't find them)
venues["sg_matched"] = venues["sg_venue_id"].notna()
unmatched = venues[~venues["sg_matched"]].sort_values("n_shows", ascending=False)
share = shows["venue_key"].isin(unmatched["venue_key"]).mean()
print(f"{len(unmatched):,} venues not matched ({share:.0%} of shows)")
print("Busiest unmatched venues (worth a quick look for typos or wrong cities):")
display(unmatched[["venue", "city", "n_shows", "median_max_listings"]].head(10))
# These stay in: their SeatGeek columns (state, capacity, etc.) are just missing.
# median_max_listings is still there as a rough stand-in for venue size.

In [ ]:
# 4. Non-US shows
KEEP_UNKNOWN_COUNTRY = True   # True = keep shows whose venue SeatGeek couldn't place
NON_US_CITIES = ["London", "Montreal", "Toronto", "Vancouver", "Manchester", "Glasgow",
                 "Dublin", "Mexico City", "Calgary", "Edmonton", "Ottawa", "Winnipeg"]

shows = shows.merge(venues[["venue_key", "is_us", "country"]], on="venue_key", how="left")
is_us = shows["is_us"].map({True: True, False: False, "True": True, "False": False})
unknown = is_us.isna()
foreign_city = shows["city"].isin(NON_US_CITIES)   # backup check for unmatched venues

keep = (is_us == True) | (unknown & ~foreign_city & KEEP_UNKNOWN_COUNTRY)
dropped = shows[~keep]
print(f"Dropping {len(dropped):,} non-US shows; keeping {keep.sum():,}")
print("Most common dropped cities:")
print(dropped["city"].value_counts().head(10).to_string())
shows = shows[keep].drop(columns=["is_us"]).reset_index(drop=True)

In [ ]:
# 5. Extreme prices
TARGET = "price_median"   # or "price_min": group decision
PRICE_CAP = 2000          # drop shows priced above this ($)

too_high = shows[TARGET] > PRICE_CAP
print(f"{too_high.sum()} shows with {TARGET} above ${PRICE_CAP:,} ({too_high.mean():.2%} of shows):")
display(shows.loc[too_high, ["name", "venue", "city", "listings", "price_min", "price_median"]]
             .sort_values(TARGET, ascending=False).head(20))
shows = shows[~too_high].reset_index(drop=True)

print(f"\nAfter cleaning: {len(shows):,} shows")
print(shows[TARGET].describe(percentiles=[.05, .25, .5, .75, .95]).round(0).to_string())